# Fine-tuning TinyLlama with QLoRA

This notebook demonstrates fine-tuning `TinyLlama-1.1B-Chat` on a medical Q&A dataset using QLoRA (4-bit quantization + LoRA adapters).

**Technique:** QLoRA — only 0.2% of parameters are trained, drastically reducing memory and compute requirements.

**Platform:** Kaggle (Tesla T4, 16GB VRAM)

## 1. Install Dependencies

In [ ]:
!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    bitsandbytes \
    trl==0.9.6 \
    accelerate==0.34.0 \
    datasets==2.19.0

## 2. Load and Prepare Dataset

Using [MedQuad](https://huggingface.co/datasets/keivalya/MedQuad-MedicalQnADataset) — a medical Q&A dataset. We filter for relevant question types and format examples in Alpaca instruction format.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
print(f"Total examples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

In [ ]:
# Filter for relevant question types
relevant_types = ["symptoms", "causes", "treatment", "prevention", "information", "exams and tests"]

filtered_dataset = dataset.filter(lambda x: x['qtype'] in relevant_types)
print(f"Original: {len(dataset)} examples")
print(f"Filtered: {len(filtered_dataset)} examples")

In [ ]:
# Format into Alpaca instruction format
def format_medical_prompt(example):
    return {
        "text": f"""### Instruction:\n{example['Question']}\n\n### Response:\n{example['Answer']}"""
    }

hf_dataset = filtered_dataset.select(range(990))
hf_dataset = hf_dataset.map(format_medical_prompt)

print(f"Sample example:")
print(hf_dataset[0]['text'][:300])

In [ ]:
# Add 10 custom examples and combine
from datasets import Dataset, concatenate_datasets

custom_data = [
    {"text": "### Instruction:\nI have a high fever and chills. What could be causing this?\n\n### Response:\nA high fever with chills can indicate several conditions including influenza, bacterial infection, malaria, or pneumonia. Monitor your temperature and seek immediate medical attention if fever exceeds 39.5°C or persists more than 3 days."},
    {"text": "### Instruction:\nWhat are the warning signs of a heart attack?\n\n### Response:\nWarning signs include chest pain or pressure, pain radiating to arm or jaw, shortness of breath, cold sweats, nausea, and lightheadedness. This is a medical emergency — call emergency services immediately."},
    {"text": "### Instruction:\nI have been experiencing headaches every day for a week. Should I be worried?\n\n### Response:\nDaily headaches lasting a week warrant medical evaluation. Possible causes include tension headaches, migraines, hypertension, or in rare cases more serious conditions. Please consult a doctor, especially if accompanied by fever, vision changes, or neck stiffness."},
    {"text": "### Instruction:\nWhat are the symptoms of appendicitis?\n\n### Response:\nAppendicitis symptoms include sudden pain starting around the navel and moving to the lower right abdomen, nausea, vomiting, fever, and loss of appetite. This is a medical emergency requiring immediate hospital attention."},
    {"text": "### Instruction:\nI have a persistent cough and night sweats. What should I do?\n\n### Response:\nPersistent cough with night sweats can indicate tuberculosis, lymphoma, or other serious conditions. Please seek immediate medical evaluation. Do not ignore these symptoms as early diagnosis is critical."},
    {"text": "### Instruction:\nWhat are the early signs of diabetes?\n\n### Response:\nEarly signs of diabetes include frequent urination, excessive thirst, unexplained weight loss, blurred vision, slow healing wounds, and frequent infections. Consult a doctor for blood glucose testing if you experience these symptoms."},
    {"text": "### Instruction:\nI feel dizzy and my heart is racing. What could this mean?\n\n### Response:\nDizziness with rapid heartbeat can indicate arrhythmia, anxiety, dehydration, low blood sugar, or anemia. If symptoms are severe or accompanied by chest pain, seek emergency care immediately."},
    {"text": "### Instruction:\nWhat are the symptoms of pneumonia?\n\n### Response:\nPneumonia symptoms include cough with phlegm, fever, chills, shortness of breath, chest pain when breathing, and fatigue. Elderly patients and young children are at higher risk. Medical evaluation and treatment are essential."},
    {"text": "### Instruction:\nI have severe abdominal pain after eating. What could it be?\n\n### Response:\nSevere abdominal pain after eating may indicate gallstones, pancreatitis, peptic ulcer, or gastritis. Please consult a doctor promptly, especially if pain is severe or accompanied by vomiting."},
    {"text": "### Instruction:\nWhat are the signs of a stroke?\n\n### Response:\nRemember FAST: Face drooping, Arm weakness, Speech difficulty, Time to call emergency services. Other signs include sudden severe headache, vision problems, and loss of balance. Stroke is a medical emergency — every minute counts."},
]

custom_dataset = Dataset.from_list(custom_data)
hf_text_only = hf_dataset.select_columns(['text'])
final_dataset = concatenate_datasets([hf_text_only, custom_dataset]).shuffle(seed=42)

print(f"Final dataset size: {len(final_dataset)} examples")

## 3. Load Model with 4-bit Quantization

We use `BitsAndBytesConfig` to load TinyLlama in 4-bit NF4 format, reducing VRAM usage from ~4.4GB to ~0.7GB.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. Configure LoRA Adapters

LoRA injects small trainable matrices into the attention layers (`q_proj`, `v_proj`). With `r=16`, we train only **2.25M parameters** out of 1.1B total (0.2%).

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./medical-chatbot-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch size = 8
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_steps=125,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    evaluation_strategy="no",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=final_dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
)

trainer.train()

## 6. Save and Upload Adapters to Hugging Face

In [ ]:
save_path = "./medical-chatbot-adapters"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

import os
for f in os.listdir(save_path):
    size = os.path.getsize(f"{save_path}/{f}")
    print(f"  {f}: {size / 1e6:.2f} MB")

In [ ]:
from huggingface_hub import HfApi

# Replace with your Hugging Face write token
my_token = "YOUR_HF_TOKEN"

api = HfApi(token=my_token)
api.create_repo(repo_id="YOUR_USERNAME/medical-chatbot-tinyllama", exist_ok=True, private=True)
api.upload_folder(folder_path=save_path, repo_id="YOUR_USERNAME/medical-chatbot-tinyllama", token=my_token)

print("Adapters uploaded to Hugging Face.")

## 7. Results

| Metric | Value |
|--------|-------|
| Base model | TinyLlama-1.1B-Chat-v1.0 |
| Trainable parameters | 2,252,800 (0.2%) |
| Training examples | 1,000 |
| Epochs | 3 |
| Final loss | ~1.44 |
| Training time | ~15 min (Tesla T4) |

Adapters hosted at: [SafiaSidimoussa/medical-chatbot-tinyllama](https://huggingface.co/SafiaSidimoussa/medical-chatbot-tinyllama)